<a href="https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/%20%20w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SunSpot-Tech/Flyrank_Internship/blob/main/work/notebooks/w06_validation_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb, os, pandas as pd, numpy as np
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ["HF_TOKEN"]}');
""")

FACT = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet"
DIM  = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id, f.client_hash_id,
        DATE '2026-03-16' - d.content_updated_date AS days_since_last_update,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_impressions ELSE 0 END) AS impressions_first_half,
        SUM(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END) AS clicks_first_half,
        AVG(CASE WHEN f.report_date < DATE '2026-03-16' THEN f.gsc_avg_position END) AS avg_position_first_half,
        SUM(CASE WHEN f.report_date >= DATE '2026-03-16' THEN f.gsc_clicks ELSE 0 END) AS clicks_second_half
    FROM read_parquet('{FACT}', hive_partitioning=1) f
    JOIN read_parquet('{DIM}') d ON f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-03' AND f.gsc_data_available IS TRUE
    GROUP BY f.content_hash_id, f.client_hash_id, d.content_updated_date
""").df()

feature_frame['days_since_last_update'] = feature_frame['days_since_last_update'].where(
    feature_frame['days_since_last_update'] >= 0, np.nan
)
feature_frame['is_declining'] = (feature_frame.clicks_second_half < feature_frame.clicks_first_half).astype(int)
feature_frame['ctr_first_half'] = feature_frame.clicks_first_half / feature_frame.impressions_first_half.replace(0, np.nan)

pos_bins = [0, 3, 6, 10, 20, 1000]
pos_labels = ['1-3', '4-6', '7-10', '11-20', '20+']
feature_frame['position_tier'] = pd.cut(feature_frame.avg_position_first_half, bins=pos_bins, labels=pos_labels)
tier_avg = feature_frame.groupby('position_tier', observed=True)['ctr_first_half'].mean()
feature_frame['tier_avg_ctr'] = feature_frame['position_tier'].map(tier_avg).astype(float)
feature_frame['ctr_gap'] = feature_frame['ctr_first_half'] < feature_frame['tier_avg_ctr']

features = ['impressions_first_half', 'clicks_first_half', 'avg_position_first_half',
            'ctr_first_half', 'days_since_last_update']

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

print(feature_frame.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(176738, 12)


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

model_df = feature_frame.dropna(subset=['avg_position_first_half']).copy()
X = model_df[features].fillna(-1)
y = model_df['is_declining']
groups = model_df['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_df = model_df.iloc[test_idx].copy()

print(f"Train: {len(X_train)} rows, {groups.iloc[train_idx].nunique()} clients")
print(f"Test:  {len(X_test)} rows, {groups.iloc[test_idx].nunique()} clients")

Train: 110403 rows, 30 clients
Test:  41578 rows, 14 clients


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def assign_reason(row):
    is_stale = pd.notna(row['days_since_last_update']) and row['days_since_last_update'] >= 180
    is_visible = row['impressions_first_half'] >= 500
    has_ctr_gap = row['ctr_gap']
    if is_stale and is_visible and has_ctr_gap:
        return 2
    elif is_stale and is_visible:
        return 1
    elif has_ctr_gap:
        return 1
    else:
        return 0

test_df['baseline_flag_score'] = test_df.apply(assign_reason, axis=1) * test_df['impressions_first_half']

tree = DecisionTreeClassifier(max_depth=4, class_weight='balanced', random_state=42)
tree.fit(X_train, y_train)
tree_scores = tree.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

results = []
for k in (20, 50, 100):
    results.append({
        'K': k,
        'Baseline rule': round(precision_at_k(test_df['baseline_flag_score'], y_test, k), 3),
        'Decision Tree': round(precision_at_k(tree_scores, y_test, k), 3),
        'Random Forest': round(precision_at_k(rf_scores, y_test, k), 3),
    })
print(pd.DataFrame(results))

     K  Baseline rule  Decision Tree  Random Forest
0   20           0.30           0.95           1.00
1   50           0.32           0.90           0.92
2  100           0.42           0.86           0.88


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding A: "What Predicts Health?" (ML Appendix, Random Forest feature
importance for Health Score)**

The paper reports Average Position (43%), Impressions (32%), and Scroll
Depth (15%) as the top predictors of Health Score.

**My methodology question — where does the label come from?**
Health Score is explicitly defined earlier in the paper as *Impressions
(30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)* — a
composite built directly from three of the same features the model then
"discovers" as top predictors. This isn't a criticism unique to this
paper; it's the same trap I found in my own Week 3 notebook, where
`trend_pct` and `trend_direction` were label-derived and had to be
excluded as features. The paper does disclose this ("the target itself
is partly constructed from some of these inputs, so importance is
descriptive rather than causal") — which is exactly the right move, and
the one I'd want to make myself. My constructive question would simply
be: given the circularity, would it be worth also reporting feature
importance against a truly external outcome (e.g., 30-day click growth)
so readers get one fully causal-safe importance ranking alongside the
descriptive one?

**Finding B: "The Freshness Multiplier" (Finding #4)**

The paper reports a 283:1 growth-to-decline ratio for the 361+ day
freshness bucket, compared to 7.88:1 for the 31-90 day bucket.

**My methodology question — does the validation design support the
claim?**
The paper itself discloses that the 361+ bucket's 283:1 ratio comes from
"283 growing pages versus only 1 declining page" — meaning the ratio is
extremely sensitive to that single denominator (if 2 pages had declined
instead of 1, the ratio would halve to ~141:1). This is a small-sample
instability, not a data error, and the paper is careful to say the
number "should not be treated as a headline decay proof point." My
question, in the same spirit the paper models: given how sensitive a
283:1 ratio is to one single data point, would a confidence interval or
a minimum-bucket-size footnote (matching the "min n=50" rule stated
elsewhere in the Methodology section) make this instability visible at
the moment a reader first sees the number, rather than only in the
caption below it?

**Why these two, and how this connects to my own model:** both findings
are ones the paper *itself* already flags as needing care — I'm not
catching an oversight, I'm practicing the same "attack your own model"
discipline the paper visibly applies to itself. That's the standard I
try to hold Section 2-4 below to as well.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
from sklearn.model_selection import train_test_split

base_rate = y.mean()
print(f"Base rate (share of pages actually declining): {base_rate:.3f}")

X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
rf_naive = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight='balanced', random_state=42)
rf_naive.fit(X_train_naive, y_train_naive)
naive_scores = rf_naive.predict_proba(X_test_naive)[:, 1]

# grouped scores already computed above as rf_scores / y_test
before_after = []
for k in (20, 50, 100):
    before_after.append({
        'K': k,
        'Base rate': round(base_rate, 3),
        'Naive random split (BEFORE)': round(precision_at_k(naive_scores, y_test_naive, k), 3),
        'Client-grouped split (AFTER)': round(precision_at_k(rf_scores, y_test, k), 3),
    })
print(pd.DataFrame(before_after))

Base rate (share of pages actually declining): 0.191
     K  Base rate  Naive random split (BEFORE)  Client-grouped split (AFTER)
0   20      0.191                         0.85                          1.00
1   50      0.191                         0.92                          0.92
2  100      0.191                         0.93                          0.88


| K   | Base rate | Naive random split (BEFORE) | Client-grouped split (AFTER) |
|-----|-----------|------------------------------|-------------------------------|
| 20  | 0.191     | 0.85                         | 1.00                          |
| 50  | 0.191     | 0.92                         | 0.92                          |
| 100 | 0.191     | 0.93                         | 0.88                          |

**Honest read:** the pattern is mixed, not a clean story in either
direction. At K=20, the grouped (honest) split scores *higher* than
naive (1.00 vs 0.85) — the opposite of what the leakage-hunting skill
predicts as the default expectation ("random split lets the model
memorize the group and fake skill", implying naive should score
higher). At K=50 the two are tied. At K=100, the direction flips to what
the skill predicts: naive scores higher than grouped (0.93 vs 0.88) —
a 5-point gap consistent with some degree of client-level memorization
inflating the naive split's score once the ranking moves past the very
top, noisiest rows.

**What this likely means:** at low K (20), the ranking is dominated by
the small number of extreme, easy-to-rank rows — including the
floor-effect rows flagged in Week 5, where `clicks_first_half = 0`
mechanically guarantees the label. Those rows are easy for *any* split
to rank correctly, which is probably why K=20 doesn't show the expected
gap. As K grows to 100 and the ranking has to reach further into
genuinely harder, more ambiguous rows, the naive split's advantage from
client leakage becomes visible — a 0.93 vs 0.88 gap, meaning roughly
5 of the top 100 "declining" calls in the naive split may be right only
because the model partially memorized which clients tend to decline,
not because it learned a client-independent pattern.

**Honest conclusion:** the grouped split is the correct one to report
going forward (0.88 at K=100, not 0.93), and the K=100 gap is real,
measurable evidence that a random split would have overstated
performance by about 5 percentage points at this K. The K=20 result
should not be used to claim "grouping doesn't matter" — it's likely
just swamped by the separate floor-effect issue, not evidence against
leakage.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
X_leaky = model_df[features + ['clicks_second_half']].fillna(-1)
leaky_tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leaky, y)
leaky_scores = leaky_tree.predict_proba(X_leaky)[:, 1]

print(f"Base rate: {base_rate:.3f}")
print(f"Leaky (with clicks_second_half) Precision@20: {precision_at_k(leaky_scores, y, 20):.3f}")
print(f"Honest (grouped, Random Forest) Precision@20: {precision_at_k(rf_scores, y_test, 20):.3f}")

Base rate: 0.191
Leaky (with clicks_second_half) Precision@20: 1.000
Honest (grouped, Random Forest) Precision@20: 1.000


## 3. Leakage Audit

Base rate: 0.191
Leaky (with `clicks_second_half`) Precision@20: 1.000
Honest (grouped, Random Forest) Precision@20: 1.000

**This result is inconclusive, not clean.** The skill's own verification
method says: "if [the score] doesn't [jump toward 1.0], your test
harness itself is broken." Here, the honest model is *already* at 1.000
before adding the leaky feature — there's no room left for a jump, so
this test cannot distinguish "no leakage" from "already-saturated
metric." A precision ceiling at K=20 masks whatever the leaky feature
would otherwise reveal.

**Correct fix, not yet run:** re-test at a K large enough that the
honest model is below 1.0 (e.g., K=200 or K=500), where a genuine gap
between honest and leaky scores could actually show up. Reporting
"no leakage found" from this test as currently run would be exactly the
kind of overclaiming Section 4 is meant to catch — so I'm flagging it
as unresolved rather than declaring it clean.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**My boldest sentence, verbatim from Week 5:** "Both models clearly beat
the Week 4 baseline rule at every K — the gap is large (e.g. at K=20:
0.30 baseline vs 0.95-1.00 for the models)."

**Rewritten in safe language, incorporating what this week's audit
found:** "In this observed March 2026 sample, both the Decision Tree and
Random Forest showed substantially higher measured Precision@K than the
Week 4 baseline rule under a client-grouped validation split (e.g.,
~0.95–1.00 vs 0.30 at K=20). This should be read as directional,
decision-support evidence favoring a model over the hand-written rule —
not as a guaranteed performance figure. Two open caveats limit how far
this claim can be pushed: first, Week 5's own analysis found the
top-line Precision@20 is likely inflated by a floor effect in low-volume
pages (`clicks_first_half = 0` mechanically forces `is_declining = 0`);
second, this week's attempt to compare a naive random split against the
honest client-grouped split was inconclusive, because both splits
saturate near a perfect score at low K — meaning I cannot yet confirm
how much of the reported precision reflects genuine client-independent
skill versus the same floor-effect ceiling. A trustworthy version of
this claim would need both issues resolved first: filtering out
low-volume floor-effect rows, and re-comparing splits at a K large
enough to avoid ceiling saturation."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.